In [1]:
import pandas as pd
import pickle

In [2]:
books = pd.read_csv("data/Books.csv", low_memory=False)

In [3]:
books.head(1)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...


In [4]:
books.shape

(271360, 8)

In [5]:
books.isna().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [6]:
books = books.dropna()

In [7]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='str')

In [8]:
books.duplicated().sum()

np.int64(0)

In [9]:
books = books.drop(columns=["Image-URL-S", "Image-URL-M"])

books.head(1)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...


In [10]:
filter_books = books[["Book-Title", "Year-Of-Publication", "Book-Author", "Publisher"]]
filter_books.head()

,Book-Title,Year-Of-Publication,Book-Author,Publisher
0,Classical Mythology,2002,Mark P. O. Morford,Oxford University Press
1,Clara Callan,2001,Richard Bruce Wright,HarperFlamingo Canada
2,Decision in Normandy,1991,Carlo D'Este,HarperPerennial
3,Flu: The Story of the Great Influenza Pandemic...,1999,Gina Bari Kolata,Farrar Straus Giroux
4,The Mummies of Urumchi,1999,E. J. W. Barber,W. W. Norton &amp; Company


In [11]:
filter_books.info()

<class 'pandas.DataFrame'>
Index: 271353 entries, 0 to 271359
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   Book-Title           271353 non-null  str  
 1   Year-Of-Publication  271353 non-null  str  
 2   Book-Author          271353 non-null  str  
 3   Publisher            271353 non-null  str  
dtypes: str(4)
memory usage: 10.4 MB


In [12]:
filter_books["Year-Of-Publication"] = (
    filter_books["Year-Of-Publication"]
    .apply(lambda x: [x])
    .apply(lambda x: [i.replace(" ", "") for i in x])
)
filter_books["Book-Author"] = (
    filter_books["Book-Author"]
    .apply(lambda x: [x])
    .apply(lambda x: [i.replace(" ", "") for i in x])
)
filter_books["Publisher"] = (
    filter_books["Publisher"]
    .apply(lambda x: [x])
    .apply(lambda x: [i.replace(" ", "") for i in x])
)

filter_books.head(1)

,Book-Title,Year-Of-Publication,Book-Author,Publisher
0,Classical Mythology,[2002],[MarkP.O.Morford],[OxfordUniversityPress]


In [13]:
filter_books["Tags"] = (
    filter_books["Book-Author"]
    + filter_books["Publisher"]
    + filter_books["Year-Of-Publication"]
)

filter_books.head(1)

,Book-Title,Year-Of-Publication,Book-Author,Publisher,Tags
0,Classical Mythology,[2002],[MarkP.O.Morford],[OxfordUniversityPress],"[MarkP.O.Morford, OxfordUniversityPress, 2002]"


In [14]:
final_books = filter_books[["Book-Title", "Tags"]]
final_books.head()

,Book-Title,Tags
0,Classical Mythology,"[MarkP.O.Morford, OxfordUniversityPress, 2002]"
1,Clara Callan,"[RichardBruceWright, HarperFlamingoCanada, 2001]"
2,Decision in Normandy,"[CarloD'Este, HarperPerennial, 1991]"
3,Flu: The Story of the Great Influenza Pandemic...,"[GinaBariKolata, FarrarStrausGiroux, 1999]"
4,The Mummies of Urumchi,"[E.J.W.Barber, W.W.Norton&amp;Company, 1999]"


In [15]:
final_books["Tags"] = final_books["Tags"].apply(lambda x: " ".join(x))
final_books["Tags"] = final_books["Tags"].apply(lambda x: x.lower())
final_books["Tags"].head()

0      markp.o.morford oxforduniversitypress 2002
1    richardbrucewright harperflamingocanada 2001
2                carlod'este harperperennial 1991
3          ginabarikolata farrarstrausgiroux 1999
4        e.j.w.barber w.w.norton&amp;company 1999
Name: Tags, dtype: str

In [16]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()


def stem(txt):
    ls = []
    for i in txt.split():
        ls.append(ps.stem(i))
    return " ".join(ls)

In [17]:
final_books["Tags"] = final_books["Tags"].apply(stem)
final_books["Tags"]

0                markp.o.morford oxforduniversitypress 2002
1              richardbrucewright harperflamingocanada 2001
2                             carlod'est harperperenni 1991
3                    ginabarikolata farrarstrausgiroux 1999
4                    e.j.w.barb w.w.norton&amp;compani 1999
                                ...                        
271355         pauladanzig randomhousechildrenspub(mm) 1988
271356                            terisloat duttonbook 1991
271357                christinewick harpersanfrancisco 2004
271358                     plato oxforduniversitypress 1996
271359    christopherbiffl mcgraw-hillhumanities/socials...
Name: Tags, Length: 271353, dtype: str

In [18]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000, stop_words="english")

In [19]:
final_books = final_books.reset_index(drop=True)
vectors = vectorizer.fit_transform(final_books["Tags"])

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# similarity = cosine_similarity(vectors)


def recommend(book_title, top_n=5):
    # Find the index of the book
    book_idx = final_books[final_books["Book-Title"] == book_title].index[0]

    # Compute similarity of just this 1 book against all 271,353 books -> shape (1, 271353)
    sim_scores = cosine_similarity(vectors[book_idx], vectors).flatten()  # type: ignore

    # Get indices of the top_n most similar books (excluding the book itself)
    similar_indices = np.argsort(sim_scores)[::-1][1 : top_n + 1]

    return books.iloc[similar_indices]

In [21]:
book = {
    "title": recommend("Tell Me This Isn't Happening")["Book-Title"].tolist(),
    "author": recommend("Tell Me This Isn't Happening")["Book-Author"].tolist(),
    "publisher": recommend("Tell Me This Isn't Happening")["Publisher"].tolist(),
    "img": recommend("Tell Me This Isn't Happening")["Image-URL-L"].tolist(),
}

print(
    f"Title: {book["title"][0]} - Author: {book['author'][0]} - Publisher: {book['publisher'][0]} - Img: {book['img'][0]}"
)

Title: Stay True: Short Stories for Strong Girls (Apple Signature) - Author: Marilyn Singer - Publisher: Scholastic - Img: http://images.amazon.com/images/P/0590360337.01.LZZZZZZZ.jpg


In [22]:
# Pkl

pickle.dump(books, open("books.pkl", "wb"))
pickle.dump(vectors, open("vectors.pkl", "wb"))

In [23]:
recommend("Tell Me This Isn't Happening")

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-L
48963,0590360337,Stay True: Short Stories for Strong Girls (App...,Marilyn Singer,1999,Scholastic,http://images.amazon.com/images/P/0590360337.0...
35300,0590972162,The Girl Who Chased Away Sorrow: The Diary of ...,Ann Warren Turner,1999,Scholastic,http://images.amazon.com/images/P/0590972162.0...
60709,0439104270,Adam Sandler: Not Too Shabby!,Jon Salem,1999,Scholastic,http://images.amazon.com/images/P/0439104270.0...
251504,0439137004,Science in a bottle,Sandra Markle,1999,Scholastic,http://images.amazon.com/images/P/0439137004.0...
55042,0590483137,"Black Hands, White Sails: The Story of African...",Pat McKissack,1999,Scholastic,http://images.amazon.com/images/P/0590483137.0...
